### Imports

In [ ]:
import pandas as pd
from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer

#### read/load dataset

In [2]:

DATA_PATH = Path("twcs.csv")
df = pd.read_csv(DATA_PATH)

#### Dataset overview

In [3]:
print("\n===== DATASET OVERVIEW =====")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumns:")
for column in df.columns:
    print(f"  - {column}")



===== DATASET OVERVIEW =====
Rows: 2,811,774
Columns: 7
Memory usage: 557.84 MB

Columns:
  - tweet_id
  - author_id
  - inbound
  - created_at
  - text
  - response_tweet_id
  - in_response_to_tweet_id


#### Missing Values

In [4]:
print("\n===== MISSING VALUES =====")
missing = df.isnull().sum()

for column, count in missing.items():
    percentage = count / len(df) * 100
    print(f"{column:30} {count:10,} ({percentage:.2f}%)")


===== MISSING VALUES =====
tweet_id                                0 (0.00%)
author_id                               0 (0.00%)
inbound                                 0 (0.00%)
created_at                              0 (0.00%)
text                                    0 (0.00%)
response_tweet_id               1,040,629 (37.01%)
in_response_to_tweet_id           794,335 (28.25%)


#### Inbound/Outbound

In [5]:
print("\n===== INBOUND / OUTBOUND =====")
print(df["inbound"].value_counts())
print("\nPercentage:")
print(df["inbound"].value_counts(normalize=True).mul(100).round(2))


===== INBOUND / OUTBOUND =====
inbound
True     1537843
False    1273931
Name: count, dtype: int64

Percentage:
inbound
True     54.69
False    45.31
Name: proportion, dtype: float64


#### Unique Authors

In [6]:
print("\n===== UNIQUE AUTHORS =====")
print(f"Unique authors: {df['author_id'].nunique():,}")


===== UNIQUE AUTHORS =====
Unique authors: 702,777


#### Top Authors

In [7]:
print("\n===== TOP AUTHORS =====")
print(df["author_id"].value_counts().head(30))


===== TOP AUTHORS =====
author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
Name: count, dtype: int64


#### Candidate Support Accounts

In [8]:
top_authors = df["author_id"].value_counts().head(30)

for author, count in top_authors.items():

    brand_tweets = df[df["author_id"] == author]

    inbound = brand_tweets["inbound"].sum()
    outbound = (~brand_tweets["inbound"]).sum()

    print(
        f"{author:25} "
        f"total={count:8,} "
        f"inbound={inbound:8,} "
        f"outbound={outbound:8,}"
        )

AmazonHelp                total= 169,840 inbound=       0 outbound= 169,840
AppleSupport              total= 106,860 inbound=       0 outbound= 106,860
Uber_Support              total=  56,270 inbound=       0 outbound=  56,270
SpotifyCares              total=  43,265 inbound=       0 outbound=  43,265
Delta                     total=  42,253 inbound=       0 outbound=  42,253
Tesco                     total=  38,573 inbound=       0 outbound=  38,573
AmericanAir               total=  36,764 inbound=       0 outbound=  36,764
TMobileHelp               total=  34,317 inbound=       0 outbound=  34,317
comcastcares              total=  33,031 inbound=       0 outbound=  33,031
British_Airways           total=  29,361 inbound=       0 outbound=  29,361
SouthwestAir              total=  28,977 inbound=       0 outbound=  28,977
VirginTrains              total=  27,817 inbound=       0 outbound=  27,817
Ask_Spectrum              total=  25,860 inbound=       0 outbound=  25,860
XboxSupport 

#### Response relations

In [9]:
print("\n===== RESPONSE RELATIONSHIPS =====")
print(
    f"Tweets with response_tweet_id: "
    f"{df['response_tweet_id'].notna().sum():,}"
)

print(
    f"Tweets with in_response_to_tweet_id: "
    f"{df['in_response_to_tweet_id'].notna().sum():,}"
)


===== RESPONSE RELATIONSHIPS =====
Tweets with response_tweet_id: 1,771,145
Tweets with in_response_to_tweet_id: 2,017,439


#### Date Range

#### Normalize tweet Ids

In [10]:
print("\n===== DATA TYPES BEFORE NORMALIZATION =====")
print(df[[
    "tweet_id",
    "response_tweet_id",
    "in_response_to_tweet_id"
]].dtypes)


===== DATA TYPES BEFORE NORMALIZATION =====
tweet_id                     int64
response_tweet_id              str
in_response_to_tweet_id    float64
dtype: object


In [11]:
df["tweet_id"] = pd.to_numeric(
    df["tweet_id"],
    errors="coerce"
).astype("Int64")

df["in_response_to_tweet_id"] = pd.to_numeric(
    df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

In [12]:
tweet_to_author = (
    df.set_index("tweet_id")["author_id"]
)

print(f"Tweet ID mapping size: {len(tweet_to_author):,}")

Tweet ID mapping size: 2,811,774


In [13]:
sample = df[
    df["in_response_to_tweet_id"].notna()
].head(10)

sample[[
    "tweet_id",
    "author_id",
    "inbound",
    "in_response_to_tweet_id",
    "text"
]]

,tweet_id,author_id,inbound,in_response_to_tweet_id,text
0,1,sprintcare,False,3,@115712 I understand. I would like to assist y...
1,2,115712,True,1,@sprintcare and how do you propose we do that
2,3,115712,True,4,@sprintcare I have sent several private messag...
3,4,sprintcare,False,5,@115712 Please send us a Private Message so th...
4,5,115712,True,6,@sprintcare I did.
5,6,sprintcare,False,8,@115712 Can you please send us a private messa...
7,11,sprintcare,False,12,@115713 This is saddening to hear. Please shoo...
8,12,115713,True,15,@sprintcare You gonna magically change your co...
9,15,sprintcare,False,16,@115713 We understand your concerns and we'd l...
10,16,115713,True,17,@sprintcare Since I signed up with you....Sinc...


In [14]:
sample["parent_author"] = (
    sample["in_response_to_tweet_id"]
    .map(tweet_to_author)
)

sample[[
    "author_id",
    "inbound",
    "parent_author",
    "text"
]]

,author_id,inbound,parent_author,text
0,sprintcare,False,115712,@115712 I understand. I would like to assist y...
1,115712,True,sprintcare,@sprintcare and how do you propose we do that
2,115712,True,sprintcare,@sprintcare I have sent several private messag...
3,sprintcare,False,115712,@115712 Please send us a Private Message so th...
4,115712,True,sprintcare,@sprintcare I did.
5,sprintcare,False,115712,@115712 Can you please send us a private messa...
7,sprintcare,False,115713,@115713 This is saddening to hear. Please shoo...
8,115713,True,sprintcare,@sprintcare You gonna magically change your co...
9,sprintcare,False,115713,@115713 We understand your concerns and we'd l...
10,115713,True,sprintcare,@sprintcare Since I signed up with you....Sinc...


#### Support Conversations

In [15]:
support_tweets = df[
    df["author_id"] == "AmazonHelp"
].copy()

print(f"AmazonHelp tweets: {len(support_tweets):,}")

AmazonHelp tweets: 169,840


In [16]:
support_tweets["customer_author"] = (
    support_tweets["in_response_to_tweet_id"]
    .map(tweet_to_author)
)

support_tweets[[
    "tweet_id",
    "customer_author",
    "in_response_to_tweet_id",
    "text"
]].head(20)

,tweet_id,customer_author,in_response_to_tweet_id,text
181,269,115770,272,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
184,273,115770,271,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
186,275,115770,274,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
234,324,115792,325,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
321,615,115820,617,@115820 I'm sorry we've let you down! Without ...
323,618,115820,616,@115820 We'd like to take a further look into ...
326,620,115822,621,@115822 I am unable to affect your account via...
328,622,115824,624,"@115824 Hi, wir erhalten die Filme/Serien so v..."
330,625,115824,623,@115824 Wir haben zu danken. Schönen Abend noc...
332,626,115826,628,@115826 I'm sorry for the wait. You'll receive...


#### Brand Customer Statistics

In [17]:
print("\n===== AMAZONHELP CUSTOMER STATISTICS =====")

print(
    f"Support replies: "
    f"{len(support_tweets):,}"
)

print(
    f"Unique customers replied to: "
    f"{support_tweets['customer_author'].nunique():,}"
)


===== AMAZONHELP CUSTOMER STATISTICS =====
Support replies: 169,840
Unique customers replied to: 71,049


In [18]:
customer_reply_counts = (
    support_tweets
    .groupby("customer_author")
    .size()
    .sort_values(ascending=False)
)

print("\nTop customers by number of AmazonHelp replies:")
print(customer_reply_counts.head(20))


Top customers by number of AmazonHelp replies:
customer_author
158494    104
326613     84
182606     65
182213     62
322303     60
295979     60
366269     53
219482     52
120538     49
144768     47
119624     47
140964     46
366798     45
190745     45
164814     44
272837     43
167356     43
270757     43
131812     42
183363     41
dtype: int64


#### Calculate Conversation/reply length chain

In [19]:
support_tweets["parent_author"] = (
    support_tweets["in_response_to_tweet_id"]
    .map(tweet_to_author)
)

conversation_counts = (
    support_tweets
    .groupby("customer_author")
    .size()
)

print("\n===== REPLIES PER CUSTOMER =====")
print(conversation_counts.describe())


===== REPLIES PER CUSTOMER =====
count    71049.000000
mean         2.376149
std          2.571225
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        104.000000
dtype: float64


In [20]:
print("\nCustomers with 1 support reply:")
print((conversation_counts == 1).sum())

print("Customers with 2 support replies:")
print((conversation_counts == 2).sum())

print("Customers with 3+ support replies:")
print((conversation_counts >= 3).sum())

print("Customers with 5+ support replies:")
print((conversation_counts >= 5).sum())


Customers with 1 support reply:
32748
Customers with 2 support replies:
18060
Customers with 3+ support replies:
20241
Customers with 5+ support replies:
7102


#### Sample Conversations

In [21]:
customer_ids = (
    support_tweets["customer_author"]
    .dropna()
    .value_counts()
    .head(10)
    .index
)

sample_conversations = df[
    df["author_id"].isin(["AmazonHelp"]) |
    (
        df["author_id"].isin(customer_ids)
    )
].copy()

In [22]:
sample_support = support_tweets[
    support_tweets["in_response_to_tweet_id"].notna()
].head(20)

sample_support[[
    "tweet_id",
    "customer_author",
    "in_response_to_tweet_id",
    "text"
]]

,tweet_id,customer_author,in_response_to_tweet_id,text
181,269,115770,272,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
184,273,115770,271,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
186,275,115770,274,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
234,324,115792,325,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
321,615,115820,617,@115820 I'm sorry we've let you down! Without ...
323,618,115820,616,@115820 We'd like to take a further look into ...
326,620,115822,621,@115822 I am unable to affect your account via...
328,622,115824,624,"@115824 Hi, wir erhalten die Filme/Serien so v..."
330,625,115824,623,@115824 Wir haben zu danken. Schönen Abend noc...
332,626,115826,628,@115826 I'm sorry for the wait. You'll receive...


#### Text Statistics

In [23]:
df["text_length"] = df["text"].str.len()

print("\n===== TEXT LENGTH =====")
print(df["text_length"].describe())


===== TEXT LENGTH =====
count    2.811774e+06
mean     1.138897e+02
std      5.234562e+01
min      1.000000e+00
25%      7.800000e+01
50%      1.150000e+02
75%      1.390000e+02
max      5.130000e+02
Name: text_length, dtype: float64


In [24]:
amazon_text = df[
    df["author_id"] == "AmazonHelp"
]["text_length"]

print("\n===== AMAZONHELP TEXT LENGTH =====")
print(amazon_text.describe())


===== AMAZONHELP TEXT LENGTH =====
count    169840.000000
mean        123.963895
std          46.174739
min           7.000000
25%          98.000000
50%         123.000000
75%         135.000000
max         305.000000
Name: text_length, dtype: float64


In [25]:
amazon_responses = (
    df[df["author_id"] == "AmazonHelp"]["text"]
)

response_counts = (
    amazon_responses
    .value_counts()
)

print("\n===== MOST COMMON AMAZONHELP RESPONSES =====")
print(response_counts.head(20))


===== MOST COMMON AMAZONHELP RESPONSES =====
text
@140964 Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. ^SQ                                                                                                3
@374403 Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. ^SQ                                                                                                3
@122920 It sounds like it's time for an A-to-Z Guarantee Claim. Here's the instructions to submit a claim: https://t.co/8wCudRz9ei ^MJ                                                                                            2
@169501 you the lowest price, may result in fluctuations in our prices over time. 2/2 ^PS                                                                                                                                         2
@169501 Pricing and offers are decisi

#### Repeated Support Responses

In [26]:
amazon_responses = (
    df[df["author_id"] == "AmazonHelp"]["text"]
)

response_counts = (
    amazon_responses
    .value_counts()
)

print("\n===== MOST COMMON AMAZONHELP RESPONSES =====")
print(response_counts.head(20))


===== MOST COMMON AMAZONHELP RESPONSES =====
text
@140964 Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. ^SQ                                                                                                3
@374403 Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. ^SQ                                                                                                3
@122920 It sounds like it's time for an A-to-Z Guarantee Claim. Here's the instructions to submit a claim: https://t.co/8wCudRz9ei ^MJ                                                                                            2
@169501 you the lowest price, may result in fluctuations in our prices over time. 2/2 ^PS                                                                                                                                         2
@169501 Pricing and offers are decisi

In [27]:
print(
    f"\nUnique AmazonHelp responses: "
    f"{amazon_responses.nunique():,}"
)

print(
    f"Total AmazonHelp responses: "
    f"{len(amazon_responses):,}"
)


Unique AmazonHelp responses: 169,805
Total AmazonHelp responses: 169,840


#### Candidate Brand Comparision Table

In [28]:
candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

brand_stats = []

for brand in candidate_brands:

    brand_tweets = df[
        df["author_id"] == brand
    ]

    brand_stats.append({
        "brand": brand,
        "tweets": len(brand_tweets),
        "unique_customers": brand_tweets[
            "in_response_to_tweet_id"
        ].map(tweet_to_author).nunique()
    })

brand_stats = pd.DataFrame(brand_stats)

brand_stats

,brand,tweets,unique_customers
0,AmazonHelp,169840,71049
1,AppleSupport,106860,76366
2,Uber_Support,56270,38300
3,SpotifyCares,43265,27794
4,Delta,42253,22331


In [29]:
brand_stats["tweets_per_customer"] = (
    brand_stats["tweets"] /
    brand_stats["unique_customers"]
)

brand_stats

,brand,tweets,unique_customers,tweets_per_customer
0,AmazonHelp,169840,71049,2.390463
1,AppleSupport,106860,76366,1.399314
2,Uber_Support,56270,38300,1.469191
3,SpotifyCares,43265,27794,1.556631
4,Delta,42253,22331,1.892123


#### Conversation Depth By Brand

In [30]:
candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

conversation_depth = []

for brand in candidate_brands:

    brand_tweets = df[
        df["author_id"] == brand
    ].copy()

    brand_tweets["customer_author"] = (
        brand_tweets["in_response_to_tweet_id"]
        .map(tweet_to_author)
    )

    replies_per_customer = (
        brand_tweets
        .groupby("customer_author")
        .size()
    )

    conversation_depth.append({
        "brand": brand,
        "customers": replies_per_customer.shape[0],
        "avg_replies": replies_per_customer.mean(),
        "median_replies": replies_per_customer.median(),
        "1_reply": (replies_per_customer == 1).sum(),
        "2_replies": (replies_per_customer == 2).sum(),
        "3+_replies": (replies_per_customer >= 3).sum(),
        "5+_replies": (replies_per_customer >= 5).sum(),
    })

conversation_depth = pd.DataFrame(conversation_depth)

conversation_depth

,brand,customers,avg_replies,median_replies,1_reply,2_replies,3+_replies,5+_replies
0,AmazonHelp,71049,2.376149,2.0,32748,18060,20241,7102
1,AppleSupport,76366,1.396538,1.0,56470,13422,6474,863
2,Uber_Support,38300,1.467180,1.0,27457,7125,3718,712
3,SpotifyCares,27794,1.554508,1.0,18999,5275,3520,693
4,Delta,22331,1.887466,1.0,13068,4936,4327,1306


In [31]:
amazon = df[
    df["author_id"] == "AmazonHelp"
].copy()

amazon[["text"]].sample(30, random_state=42)

,text
1580471,"@523365 Ok, please keep us posted here. ^MC"
658246,@193739 Merci pour votre commentaire 😊 Vous av...
1412806,@469296 What was advised when you contacted th...
1373951,@470927 I'm sorry for this wait. What was the ...
1469183,@495154 リンクにアクセスできますか？そちらから13桁の番号が確認いただけます。ＥＴ
2621995,@748692 Oh no! Please call or chat with us her...
39614,@127184 Thanks! We'd like to look into this wi...
2747119,@137602 I'm sorry for the recent delivery issu...
569875,"@272295 ¡Hola, Julio! Lamentamos lo ocurrido, ..."
969140,"@373702 Hola Ene-chan, ¿has recibido los pedid..."


#### Amazon response Charectistics

In [32]:
amazon = df[
    df["author_id"] == "AmazonHelp"
].copy()

print("Total AmazonHelp tweets:", len(amazon))

print(
    "Replies with a parent:",
    amazon["in_response_to_tweet_id"].notna().sum()
)

print(
    "Replies with further responses:",
    amazon["response_tweet_id"].notna().sum()
)

print(
    "Terminal AmazonHelp tweets:",
    amazon["response_tweet_id"].isna().sum()
)

Total AmazonHelp tweets: 169840
Replies with a parent: 169287
Replies with further responses: 85274
Terminal AmazonHelp tweets: 84566


In [33]:
amazon_stats = {
    "total": len(amazon),
    "has_parent": amazon["in_response_to_tweet_id"].notna().mean(),
    "has_response": amazon["response_tweet_id"].notna().mean(),
    "terminal": amazon["response_tweet_id"].isna().mean()
}

amazon_stats

{'total': 169840,
 'has_parent': np.float64(0.9967439943476213),
 'has_response': np.float64(0.5020843146490815),
 'terminal': np.float64(0.4979156853509185)}

##### Terminal Responses

In [34]:
terminal_responses = amazon[
    amazon["response_tweet_id"].isna()
]

terminal_responses[
    ["tweet_id", "in_response_to_tweet_id", "text"]
].sample(
    30,
    random_state=42
)

,tweet_id,in_response_to_tweet_id,text
755825,844175,844176,@320701 Entiendo. ¿Has contactado directamente...
1338359,1474020,1474018,@462184 Thanks for confirming! Do you care to ...
940154,1043144,1043143,@366678 Please provide your details here: http...
76708,98694,98693,@137601 I understand. We are unable to start a...
651354,441856,441855,@219784 Please don't provide your order detail...
679803,760479,760478,"@301533 You're welcome, Nisharg. Do keep us po..."
535747,602268,602269,"@224022 We’re always looking to improve, and w..."
1272718,1403247,1403246,"@446257 En ese caso, te recomiendo reportar la..."
373271,426386,421385,@216413 I'm sorry you've not received your ord...
2544620,2715170,2715171,@762197 ご質問の件についてはカスタマーサービスにてご案内させていただきますので、恐れ...


### Brand Selection

Selected brand: AmazonHelp

Reason:
AmazonHelp provides a large and conversation-rich support dataset, with
169,840 support tweets, 71,049 unique customers, an average of 2.38
support replies per customer, and 7,102 customers receiving 5+ replies.

The high conversation depth makes it suitable for studying historical
support behavior and constructing grounded response examples.

#### Build tweet lookup

In [35]:
tweet_lookup = df.set_index("tweet_id")[
    ["author_id", "inbound", "text", "created_at",
     "response_tweet_id", "in_response_to_tweet_id"]
]

print("Tweet lookup created:", len(tweet_lookup))

Tweet lookup created: 2811774


#### Converstaion Reconstruction

In [ ]:
# Function to inspect one conversation
def get_conversation(tweet_id, max_depth=20):
    """
    Walk backwards through in_response_to_tweet_id
    to reconstruct the conversation leading to a tweet.
    """

    conversation = []
    current_id = tweet_id
    visited = set()

    for _ in range(max_depth):

        if pd.isna(current_id):
            break

        if current_id in visited:
            break

        visited.add(current_id)

        if current_id not in tweet_lookup.index:
            break

        row = tweet_lookup.loc[current_id]

        conversation.append({
            "tweet_id": current_id,
            "author_id": row["author_id"],
            "inbound": row["inbound"],
            "text": row["text"],
            "created_at": row["created_at"],
            "parent_id": row["in_response_to_tweet_id"]
        })

        current_id = row["in_response_to_tweet_id"]

    conversation.reverse()

    return pd.DataFrame(conversation)

##### Test it on AmazonHelp reply

In [37]:
sample_amazon_tweet = (
    amazon[
        amazon["in_response_to_tweet_id"].notna()
    ]
    .sample(1, random_state=42)
    .iloc[0]
)

sample_amazon_tweet[[
    "tweet_id",
    "in_response_to_tweet_id",
    "text"
]]

tweet_id                                                             1768950
in_response_to_tweet_id                                              1768951
text                       @267823 And we keep updating our prices as per...
Name: 1614776, dtype: object

##### Display it in readable way

In [38]:
conversation = get_conversation(
    sample_amazon_tweet["tweet_id"]
)

conversation

,tweet_id,author_id,inbound,text,created_at,parent_id
0,1768951,267823,True,@115850 @115821 Is selling almost every electr...,Tue Oct 17 21:51:35 +0000 2017,<NA>
1,1768950,AmazonHelp,False,@267823 And we keep updating our prices as per...,Tue Oct 17 21:58:34 +0000 2017,1768951


In [39]:
for _, row in conversation.iterrows():

    speaker = "CUSTOMER" if row["inbound"] else "AMAZONHELP"

    print(f"\n[{speaker}]")
    print(row["text"])


[CUSTOMER]
@115850 @115821 Is selling almost every electronic overpiced on the name of sale. And all products are old Manufactured. #onlineshopping

[AMAZONHELP]
@267823 And we keep updating our prices as per data we receive from our sellers. Please refer here: https://t.co/yQb39uNRsX (2/2) ^SV


In [40]:
amazon_samples = (
    amazon[
        amazon["in_response_to_tweet_id"].notna()
    ]
    .sample(10, random_state=42)
)

for tweet_id in amazon_samples["tweet_id"]:

    print("\n" + "=" * 80)
    print("TWEET:", tweet_id)

    conv = get_conversation(tweet_id)

    for _, row in conv.iterrows():

        speaker = (
            "CUSTOMER"
            if row["inbound"]
            else "AMAZONHELP"
        )

        print(f"\n{speaker}:")
        print(row["text"])


TWEET: 1768950

CUSTOMER:
@115850 @115821 Is selling almost every electronic overpiced on the name of sale. And all products are old Manufactured. #onlineshopping

AMAZONHELP:
@267823 And we keep updating our prices as per data we receive from our sellers. Please refer here: https://t.co/yQb39uNRsX (2/2) ^SV

TWEET: 1190175

CUSTOMER:
#hermes #hermesuk @115830 Just received an email to say my parcel has been signed for and delivered. I've been at home for several hours and no parcel has been delivered &amp; I have definitely not signed anything. Please advise what to do next.

AMAZONHELP:
@399739 I'm sorry for the trouble with your delivery. I recommend trying these steps: https://t.co/eVVwAETSwo Please keep us posted. ^RA

CUSTOMER:
@AmazonHelp I found those steps on your website and have checked them including trying to contact Hermes. Only option was to email them which I have done.

AMAZONHELP:
@399739 Please keep us updated on the outcome, Laurence. ^WJ

CUSTOMER:
@AmazonHelp Thi

#### Create customer-support pairs
###### Creating actual conversation database

In [41]:
amazon_replies = df[
    (df["author_id"] == "AmazonHelp") &
    (df["in_response_to_tweet_id"].notna())
].copy()

amazon_replies["customer_tweet_id"] = (
    amazon_replies["in_response_to_tweet_id"]
)

print("AmazonHelp replies:", len(amazon_replies))

customer_support_pairs = amazon_replies.merge(
    df[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "created_at"
        ]
    ],
    left_on="customer_tweet_id",
    right_on="tweet_id",
    suffixes=("_support", "_customer")
)

print("Pairs created:", len(customer_support_pairs))
print(customer_support_pairs.columns.tolist())

customer_support_pairs = customer_support_pairs.rename(
    columns={
        "tweet_id_support": "support_tweet_id",
        "author_id_support": "support_author",
        "text_support": "support_text",
        "created_at_support": "support_created_at",

        "tweet_id": "customer_tweet_id",
        "author_id_customer": "customer_author",
        "inbound_customer": "customer_inbound",
        "text_customer": "customer_text",
        "created_at_customer": "customer_created_at"
    }
)

customer_support_pairs[
    [
        "customer_tweet_id",
        "customer_author",
        "customer_inbound",
        "customer_text",
        "support_tweet_id",
        "support_text"
    ]
].head(10)

AmazonHelp replies: 169287
Pairs created: 168823
['tweet_id_support', 'author_id_support', 'inbound_support', 'created_at_support', 'text_support', 'response_tweet_id', 'in_response_to_tweet_id', 'text_length', 'customer_tweet_id', 'tweet_id_customer', 'author_id_customer', 'inbound_customer', 'text_customer', 'created_at_customer']


,customer_tweet_id,customer_author,customer_inbound,customer_text,support_tweet_id,support_text
0,272,115770,True,amazonのfireTVstickが見れない😢,269,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,271,115770,True,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,274,115770,True,@AmazonHelp こちらこそありがとうございました。,275,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,325,115792,True,amazonプライムビデオ、再生エラーが多いです,324,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
4,617,115820,True,Way to drop the ball on customer service @1158...,615,@115820 I'm sorry we've let you down! Without ...
5,616,115820,True,@AmazonHelp 3 different people have given 3 di...,618,@115820 We'd like to take a further look into ...
6,621,115822,True,@115823 I want my amazon payments account CLOS...,620,@115822 I am unable to affect your account via...
7,624,115824,True,"@115825 also, beim Addams Family-Film in Prime...",622,"@115824 Hi, wir erhalten die Filme/Serien so v..."
8,623,115824,True,"@AmazonHelp Okay, danke für die Info",625,@115824 Wir haben zu danken. Schönen Abend noc...
9,628,115826,True,@115828 How about you guys figure out my Xbox ...,626,@115826 I'm sorry for the wait. You'll receive...


In [42]:
customer_support_pairs["customer_inbound"] = (
    customer_support_pairs["tweet_id_customer"]
    .map(
        df.set_index("tweet_id")["inbound"]
    )
)
customer_support_pairs["customer_inbound"].value_counts()

customer_inbound
True     168814
False         9
Name: count, dtype: int64

##### Calculate useful pair percentage

In [43]:
valid_pairs = customer_support_pairs[
    customer_support_pairs["customer_inbound"] == True
].copy()

print(f"Valid customer → AmazonHelp pairs: {len(valid_pairs):,}")
print(f"Invalid pairs removed: {(customer_support_pairs['customer_inbound'] == False).sum()}")

Valid customer → AmazonHelp pairs: 168,814
Invalid pairs removed: 9


#### customer message  quality

In [44]:
valid_pairs["customer_text_length"] = (
    valid_pairs["customer_text"].str.len()
)

valid_pairs["support_text_length"] = (
    valid_pairs["support_text"].str.len()
)

print("Customer text length:")
print(valid_pairs["customer_text_length"].describe())

print("\nSupport text length:")
print(valid_pairs["support_text_length"].describe())

Customer text length:
count    168814.000000
mean        116.589033
std          59.101826
min           7.000000
25%          72.000000
50%         118.000000
75%         145.000000
max         365.000000
Name: customer_text_length, dtype: float64

Support text length:
count    168814.000000
mean        123.936925
std          46.187720
min           7.000000
25%          98.000000
50%         123.000000
75%         135.000000
max         305.000000
Name: support_text_length, dtype: float64


In [45]:
valid_pairs[
    [
        "customer_text",
        "support_text",
        "customer_text_length"
    ]
].sort_values(
    "customer_text_length"
).head(30)

,customer_text,support_text,customer_text_length
134114,@116090,@684978 Hey! Thanks for reaching out! We won't...,7
152771,@116875,"@765126 Hola Teufel, ¿cómo podemos ayudarte? ^DA",7
155595,@115850,@777178 I'm sorry but we were unable to appreh...,7
156446,@115825,@781498 Ja? Was gibt es? Lieben Gruß ^NW,7
66485,@119625,"@350250 Hello, are you looking for any assista...",7
144293,@115850,@725323 How may we help? ^JS,7
92415,@115821,@442253 Sorry to know that the product is defe...,7
112558,@115850,"@524862 Looks like you have a question, Ashok....",7
135588,@115850,@689154 Is there anything we may assist you wi...,7
55381,@115850,@206099 Allow our support team to look into th...,7


In [46]:
valid_pairs[
    [
        "customer_text",
        "support_text",
        "customer_text_length"
    ]
].sort_values(
    "customer_text_length",
    ascending=False
).head(10)

,customer_text,support_text,customer_text_length
6325,@141810 @115821 @6643 @141811 @141812 @6644 @1...,@141809 We'd like to link this feedback to the...,365
7041,@AmazonHelp @115821 @115830 @116935 @117634 @1...,@144768 I'm sorry the refund hasn't been credi...,364
7038,@AmazonHelp @115821 @115851 @115830 @117634 @1...,@144768 I understand to your concern regarding...,350
38625,@233674 @115850 @4449 @115821 @348 @22823 @233...,@127186 All after sales services (warranty and...,337
38624,@233674 @115850 @4449 @115821 @348 @22823 @233...,@127186 They're the right ones to approach for...,337
38629,@233674 @115850 @4449 @115821 @348 @22823 @233...,@127186 Sorry to know about the concerns with ...,337
143433,@AmazonHelp @115851 @115821 @115851 pl see d i...,@722085 Please reply to the email for any furt...,333
38516,@233674 @115850 @4449 @115821 @348 @22823 @233...,@127186 Sorry to know you're facing difficulti...,329
35998,"@225822 @AmazonHelp @3375 I love Amazon, but I...",@225821 I'm sorry for your poor experience wit...,327
137549,@115851 @115850 @115821 As we can see amazon a...,@529275 The information provided over email is...,327


#### Normalise customer texts

In [76]:

# 1. Make sure we have only valid customer -> AmazonHelp pairs
valid_pairs = customer_support_pairs[
    customer_support_pairs["customer_inbound"] == True
].copy()

print("Valid pairs:", len(valid_pairs))


# 2. Define cleaning function
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove @mentions such as @115850 and @AmazonHelp
    text = re.sub(r"@\w+", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove HTML entities
    text = re.sub(r"&amp;", "and", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# 3. Clean customer text
valid_pairs["customer_text_clean"] = (
    valid_pairs["customer_text"]
    .apply(normalize_text)
)


# 4. Calculate cleaned text length
valid_pairs["clean_length"] = (
    valid_pairs["customer_text_clean"].str.len()
)


# 5. Verify
print("\nColumns created:")
print(
    valid_pairs[
        ["customer_text", "customer_text_clean", "clean_length"]
    ].head()
)

print("\nShortest messages:")
print(
    valid_pairs[
        [
            "customer_text",
            "customer_text_clean",
            "support_text",
            "clean_length"
        ]
    ]
    .sort_values("clean_length")
    .head(30)
)

Valid pairs: 168814

Columns created:
                                       customer_text  \
0                           amazonのfireTVstickが見れない😢   
1  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...   
2                      @AmazonHelp こちらこそありがとうございました。   
3                           amazonプライムビデオ、再生エラーが多いです   
4  Way to drop the ball on customer service @1158...   

                                 customer_text_clean  clean_length  
0                           amazonのfireTVstickが見れない😢            24  
1  電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんで...            51  
2                                  こちらこそありがとうございました。            17  
3                           amazonプライムビデオ、再生エラーが多いです            24  
4  Way to drop the ball on customer service so pi...            61  

Shortest messages:
                                            customer_text customer_text_clean  \
125741               @AmazonHelp  https://t.co/caIUiYCo4x                       
168611               @AmazonH

In [77]:
print("Total pairs:", len(valid_pairs))

print(
    "Empty after cleaning:",
    (valid_pairs["customer_text_clean"] == "").sum()
)

print(
    "Very short after cleaning (< 5 chars):",
    (valid_pairs["clean_length"] < 5).sum()
)

print(
    "Short after cleaning (< 15 chars):",
    (valid_pairs["clean_length"] < 15).sum()
)

Total pairs: 168814
Empty after cleaning: 1425
Very short after cleaning (< 5 chars): 2484
Short after cleaning (< 15 chars): 7052


#### Actual usable dataset

In [78]:
# Flag completely empty messages
valid_pairs["mention_only"] = (
    valid_pairs["customer_text_clean"] == ""
)

# Common low-information responses
low_info_patterns = [
    r"^\?$",
    r"^!$",
    r"^(hi|hello|hey|thanks|thank you|ok|okay|yes|no)$"
]

pattern = "|".join(low_info_patterns)

valid_pairs["low_information"] = (
    valid_pairs["customer_text_clean"]
    .str.lower()
    .str.match(pattern, na=False)
)

print("===== DATA QUALITY =====")

print("Total pairs:", len(valid_pairs))
print("Empty:", valid_pairs["mention_only"].sum())
print("Low information:", valid_pairs["low_information"].sum())

usable_pairs = valid_pairs[
    ~valid_pairs["mention_only"] &
    ~valid_pairs["low_information"]
].copy()

print("Usable pairs:", len(usable_pairs))

print(
    "Usable percentage:",
    round(len(usable_pairs) / len(valid_pairs) * 100, 2)
)

===== DATA QUALITY =====
Total pairs: 168814
Empty: 1425
Low information: 426
Usable pairs: 166963
Usable percentage: 98.9


In [79]:
valid_pairs[
    valid_pairs["mention_only"] |
    valid_pairs["low_information"]
][[
    "customer_text",
    "customer_text_clean",
    "support_text"
]].head(50)

,customer_text,customer_text_clean,support_text
30,@115821 https://t.co/Gb0beuA3IN,,@115848 I'm sorry for the trouble this has cau...
278,@AmazonHelp Yes,Yes,@118079 I hope you are willing to give us a ch...
279,@AmazonHelp Yes,Yes,@118079 I'm sorry we let you down on this orde...
401,@AmazonHelp https://t.co/saX0kAZVCW!,,@118829 Can you confirm the delivery date prov...
597,@AmazonHelp @119704 https://t.co/rUjHdeMm9G,,"@119703 However, you need to adhere to guideli..."
782,@AmazonHelp https://t.co/LzRfyE5m3q,,@120177 Thanks for the information! Who is the...
828,@AmazonHelp,,"@120259 Hey, what's up? ^AS"
854,@118919,,"@120265 Hi, how can we help you today? ^HK"
928,@118919 @119356 https://t.co/nei7OW1h0M,,@119355 Have you reached out to our support te...
1178,@AmazonHelp https://t.co/QCt7kMtESE,,"@120918 Hi, have you been in touch with the pu..."


In [80]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

usable_pairs.to_parquet(
    processed_path / "amazon_support_pairs.parquet",
    index=False
)

print("Saved:", processed_path / "amazon_support_pairs.parquet")

Saved: ..\data\processed\amazon_support_pairs.parquet
